# Level 3 — Fix wrong datatypes

Many EDA mistakes start with incorrect datatypes. I will create converted columns first so the raw values stay available for checking.

In [1]:
from pathlib import Path
import pandas as pd

# Works if I open Jupyter from the repo root or from notebooks/
data_path = Path("data/sales_raw.csv")
if not data_path.exists():
    data_path = Path("../data/sales_raw.csv")

df = pd.read_csv(data_path)
df.head(3)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
0,LN-00001,ORD-2025-25001,2025-05-13,C-1023,Lahore,Office,Whiteboard,4,6000,10%,21600.0,Card,Delivered,Store,10.0
1,LN-00002,ORD-2025-25001,2025-08-19,C-1134,Islamabad,Home,Wall Clock,5,3100,15%,13175.0,Cash,Delivered,Store,8.0
2,LN-00003,ORD-2025-25002,2025-09-10,C-1052,Rawalpindi,NaN,Desk Lamp,3,2592,0%,7776.0,Bank Transfer,Delivered,Online,6.0


In [2]:
df[["order_date", "quantity", "discount_pct", "revenue_pkr"]].dtypes

order_date       object
quantity         object
discount_pct     object
revenue_pkr     float64
dtype: object

### Dates as strings
The file includes ISO dates and a few day-first dates. There is also one invalid entry to catch.

In [3]:
df["date_parsed"] = pd.to_datetime(
    df["order_date"], errors="coerce", format="mixed", dayfirst=True
)
print("Dates that did not parse:", df["date_parsed"].isna().sum())
df.loc[df["date_parsed"].isna(), ["order_id", "order_date"]].head()

Dates that did not parse: 1


,order_id,order_date
335,ORD-2025-25168,not sure


### Quantity should be numeric
`errors="coerce"` is useful for identifying bad entries. It is not permission to forget them.

In [4]:
df["qty_num"] = pd.to_numeric(df["quantity"], errors="coerce")
print("Bad quantity entries:", df["qty_num"].isna().sum())
df.loc[df["qty_num"].isna(), ["line_id", "quantity"]].head(10)

Bad quantity entries: 2


,line_id,quantity
466,LN-00467,two
490,LN-00491,two


### Turn percentage strings into decimal values

In [5]:
df["discount_rate"] = pd.to_numeric(
    df["discount_pct"].astype("string").str.strip().str.rstrip("%"),
    errors="coerce"
) / 100

df[["discount_pct", "discount_rate"]].head(7)

,discount_pct,discount_rate
0,10%,0.1
1,15%,0.15
2,0%,0.0
3,5%,0.05
4,10%,0.1
5,0%,0.0
6,0%,0.0


### Category dtype and memory

In [6]:
print("City as object:", df["city"].memory_usage(deep=True))
print("City as category:", df["city"].astype("category").memory_usage(deep=True))
df["channel"] = df["channel"].astype("category")
print("Channel datatype:", df["channel"].dtype)

City as object: 31780
City as category: 1669
Channel datatype: category


**My note:** Conversion is only step one. I still need to investigate rows that became `NaN` or `NaT`.